In [0]:

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail_analytics"

# Leer las tablas Bronze de películas
movies = spark.table(
    f"{CATALOG}.{SCHEMA}.bronze_movies"
)

details = spark.table(
    f"{CATALOG}.{SCHEMA}.bronze_film_details"
)

posters = spark.table(
    f"{CATALOG}.{SCHEMA}.bronze_poster_paths"
)

print("Movies:", movies.columns)
print("FilmDetails:", details.columns)
print("PosterPath:", posters.columns)


Movies: ['id', 'title', 'genres', 'language', 'user_score', 'runtime_hour', 'runtime_min', 'release_date', 'vote_count']
FilmDetails: ['id', 'director', 'top_billed', 'budget_usd', 'revenue_usd']
PosterPath: ['id', 'poster_path', 'backdrop_path']


In [0]:

# ============================================================
# SILVER - LIMPIEZA Y TRANSFORMACION DE PELICULAS
# ============================================================

from pyspark.sql import functions as F

# 1. Limpiar películas
silver_movies = (
    movies
    .withColumn("id", F.col("id").cast("long"))
    .withColumn("title", F.trim(F.col("title")))
    .withColumn("genres", F.trim(F.col("genres")))
    .withColumn("language", F.trim(F.col("language")))
    .withColumn("user_score", F.col("user_score").cast("double"))
    .withColumn("runtime_hour", F.col("runtime_hour").cast("int"))
    .withColumn("runtime_min", F.col("runtime_min").cast("int"))
    .withColumn("release_date", F.to_date(F.col("release_date")))
    .withColumn("vote_count", F.col("vote_count").cast("long"))
    .dropDuplicates(["id"])
    .filter(
        F.col("id").isNotNull()
        & F.col("title").isNotNull()
        & (F.col("title") != "")
    )
)

# 2. Limpiar detalles financieros

# Limpiar detalles financieros
silver_details = (
    details
    .withColumn("id", F.col("id").cast("long"))
    .withColumn("director", F.trim(F.col("director")))
    .withColumn("top_billed", F.trim(F.col("top_billed")))

    .withColumn(
        "budget_clean",
        F.regexp_replace(
            F.trim(F.col("budget_usd")),
            r"[^0-9.-]",
            ""
        )
    )
    .withColumn(
        "revenue_clean",
        F.regexp_replace(
            F.trim(F.col("revenue_usd")),
            r"[^0-9.-]",
            ""
        )
    )
    .withColumn(
        "budget_usd",
        F.when(
            F.col("budget_clean") != "",
            F.col("budget_clean").cast("double")
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "revenue_usd",
        F.when(
            F.col("revenue_clean") != "",
            F.col("revenue_clean").cast("double")
        ).otherwise(F.lit(None).cast("double"))
    )
    .drop("budget_clean", "revenue_clean")
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
)


# 3. Unir películas y detalles
silver_movies_complete = (
    silver_movies
    .join(silver_details, on="id", how="left")
    .withColumn(
        "runtime_minutes",
        F.when(
            F.col("runtime_hour").isNotNull()
            & F.col("runtime_min").isNotNull(),
            F.col("runtime_hour") * 60 + F.col("runtime_min")
        )
    )
)

# 4. Limpiar rutas de imágenes
silver_posters = (
    posters
    .withColumn("id", F.col("id").cast("long"))
    .withColumn("poster_path", F.trim(F.col("poster_path")))
    .withColumn("backdrop_path", F.trim(F.col("backdrop_path")))
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
)

print("Películas limpias:", silver_movies.count())
print("Detalles limpios:", silver_details.count())
print("Películas con detalles:", silver_movies_complete.count())
print("Rutas de imágenes:", silver_posters.count())

display(silver_movies_complete.limit(10))
display(silver_posters.limit(10))


Películas limpias: 9718
Detalles limpios: 9718
Películas con detalles: 9718
Rutas de imágenes: 9718


id,title,genres,language,user_score,runtime_hour,runtime_min,release_date,vote_count,director,top_billed,budget_usd,revenue_usd,runtime_minutes
28,One Flew Over the Cuckoo's Nest,Drama,en,8.4,2,13,1975-11-19,10402,Miloš Forman,"Jack Nicholson, Brad Dourif, Louise Fletcher, Danny DeVito, William Redfield",3000000.0,1.08981275E8,133
29,Once Upon a Time in America,"Drama, Crime",en,8.4,3,49,1984-05-23,5368,Sergio Leone,"Robert De Niro, James Woods, Elizabeth McGovern, Treat Williams, Tuesday Weld",3.0E7,5500000.0,229
30,Gabriel's Inferno,"Romance, Drama",en,8.4,2,2,2020-05-29,2410,Tosca Musk,"Melanie Zanetti, Giulio Berruti, Kurt McKinney, Agnes Albright, Margaux Brooke",null,null,122
33,Primal: Tales of Savagery,"Action, Adventure, Animation, Drama",en,8.4,1,25,2019-11-21,315,Genndy Tartakovsky,"Aaron LaPlante, Jon Olson, Bryan Andrews, Darrick Bachman",null,null,85
93,Evangelion: 3.0+1.0 Thrice Upon a Time,"Animation, Action, Science Fiction, Drama",ja,8.2,2,35,2021-03-08,806,Kazuya Tsurumaki,"Megumi Ogata, Yuko Miyamura, Maaya Sakamoto, Megumi Hayashibara, Kotono Mitsuishi",2.97E7,9.3882453E7,155
151,8½,Drama,it,8.2,2,19,1963-02-14,2260,Federico Fellini,"Marcello Mastroianni, Anouk Aimée, Sandra Milo, Claudia Cardinale, Rossella Falk",null,null,139
171,My Friends,Comedy,it,8.1,2,20,1975-10-24,691,Mario Monicelli,"Ugo Tognazzi, Gastone Moschin, Philippe Noiret, Duilio Del Prete, Adolfo Celi",null,null,140
200,All About Eve,Drama,en,8.1,2,19,1950-11-09,1516,Joseph L. Mankiewicz,"Bette Davis, Anne Baxter, George Sanders, Celeste Holm, Gary Merrill",1400000.0,8400000.0,139
208,WALL·E,"Animation, Family, Science Fiction",en,8.1,1,38,2008-06-22,18690,Andrew Stanton,"Ben Burtt, Elissa Knight, Jeff Garlin, Fred Willard, John Ratzenberger",1.8E8,5.2131186E8,98
221,There Will Be Blood,Drama,en,8.1,2,38,2007-12-26,6563,Paul Thomas Anderson,"Daniel Day-Lewis, Paul Dano, Kevin J. O'Connor, Ciarán Hinds, Dillon Freasier",2.5E7,7.7208711E7,158


id,poster_path,backdrop_path
1,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,https://image.tmdb.org/t/p/w500/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg
2,https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,https://image.tmdb.org/t/p/w500/tmU7GeKVybMWFButWEGl2M4GeiP.jpg
3,https://image.tmdb.org/t/p/w500/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,https://image.tmdb.org/t/p/w500/kGzFbGhp99zva6oZODW5atUtnqi.jpg
4,https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,https://image.tmdb.org/t/p/w500/zb6fM1CX41D9rF9hdgclu0peUmy.jpg
5,https://image.tmdb.org/t/p/w500/ow3wq89wM8qd5X7hWKxiRfsFf9C.jpg,https://image.tmdb.org/t/p/w500/qqHQsStV6exghCM7zbObuYBiYxw.jpg
6,https://image.tmdb.org/t/p/w500/39wmItIWsg5sZMyRUHLkWBcuVCM.jpg,https://image.tmdb.org/t/p/w500/6oaL4DP75yABrd5EbC4H2zq5ghc.jpg
7,https://image.tmdb.org/t/p/w500/wTnV3PCVW5O92JMrFvvrRcV39RU.jpg,https://image.tmdb.org/t/p/w500/v9acaWVVFdZT5yAU7J2QjwfhXyD.jpg
8,https://image.tmdb.org/t/p/w500/lfRkUr7DYdHldAqi3PwdQGBRBPM.jpg,https://image.tmdb.org/t/p/w500/90ez6ArvpO8bvpyIngBuwXOqJm5.jpg
9,https://image.tmdb.org/t/p/w500/qJ2tW6WMUDux911r6m7haRef0WH.jpg,https://image.tmdb.org/t/p/w500/oOv2oUXcAaNXakRqUPxYq5lJURz.jpg
10,https://image.tmdb.org/t/p/w500/8VG8fDNiy50H4FedGwdSVUPoaJe.jpg,https://image.tmdb.org/t/p/w500/vxJ08SvwomfKbpboCWynC3uqUg4.jpg


In [0]:

# ============================================================
# SILVER - GUARDAR TABLAS DELTA
# ============================================================

(
    silver_movies_complete.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA}.silver_movies"
    )
)

(
    silver_posters.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA}.silver_posters"
    )
)

print(
    "Silver movies:",
    spark.table(
        f"{CATALOG}.{SCHEMA}.silver_movies"
    ).count()
)

print(
    "Silver posters:",
    spark.table(
        f"{CATALOG}.{SCHEMA}.silver_posters"
    ).count()
)


Silver movies: 9718
Silver posters: 9718
